# 01. 문서 요약 Tool — PDF 전처리 + `summary.py`
**Day 4**

> 수업용 문서: **`samples/Language_Models.pdf`**
> (언어 모델 개요 자료 — 긴 PDF 요약 실습에 적합)

**파이프라인:** `PDF → PyMuPDF 추출 → 전처리 → OpenAI 요약 → 저장`

---
## 0. 설치

In [ ]:
!pip install pymupdf openai python-dotenv

In [ ]:
!pip install pymupdf

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

In [ ]:
pdf_path = os.path.join(os.getcwd(),"samples/Language_models.pdf")

---
## 1. PyMuPDF로 `Language_Models.pdf` 텍스트 추출

In [ ]:
import pymupdf

doc = pymupdf.open(pdf_path)

In [ ]:
doc

In [ ]:
i=0
for page in doc:
    text = page.get_text()
    print(text)
    i+=1
    if i==5:
        break

In [ ]:
full_text=''
for page in doc:
    text = page.get_text()
    full_text+=text + '\n------------------------\n'


In [ ]:
pdf_file_name = os.path.basename(pdf_path)

In [ ]:
pdf_file_name = os.path.splitext(pdf_file_name)[0]
txt_file_path = os.path.join(os.getcwd(),'samples',f"{pdf_file_name}.txt")

In [ ]:
with open(txt_file_path, 'w', encoding='utf-8') as f:
    f.write(full_text)

---
## 2. 문서 요약

In [ ]:
api_key = os.getenv('OPENAI_API_KEY')

def summarize_txt(file_path):
    client = OpenAI(api_key=api_key)
    with open(file_path, 'r', encoding = 'utf-8') as f:
        txt = f.read()

    system_prompt = f'''
    너는 다음 글을 요약하는 봇이다. 아래 글을 읽고, 

    작성해야 하는 포맷은 다음과 같음
    # 제목

    ## 저자의 문제 인식 및 주장 (15문장 이내)

    ## 저자 소개


    ============= 이하 텍스트 ================
    {txt[:10000]}

    '''

    response = client.chat.completions.create(
        model = 'gpt-4o-mini',
        temperature = 0.1,
        messages=[
            {"role":"system","content":system_prompt},
        ]
    )

    return response.choices[0].message.content


In [ ]:
summary = summarize_txt(txt_file_path)

In [ ]:
print(summary)

## 실습 : 함수 생성

- input : pdf 파일 경로 
- output : pdf 내용을 요약한 summary.txt 파일 생성 및 summary return

- 파일명: pdf_summary.py